# Phase 1 — Univariate Bootstrap

This notebook reproduces the full Phase 1 univariate analysis:
1. **Numeric variables** — histogram + KDE + box plot for all 14 numeric columns
2. **Categorical variables** — frequency bar charts for all 5 categorical columns

All 28+ figures are saved to `phase1_univariate/figures/`.
Summary statistics are saved to `phase1_univariate/numeric_summary_stats.csv`.

> **Input:** `dataset/student_wellness_clean.csv` (produced in Phase 0)


## Part 1 — Numeric Columns

For each numeric column: histogram with KDE overlay + box plot.
Annotates mean and median with vertical dashed lines.
Prints descriptive stats (mean, median, std, skewness, kurtosis).

**Key findings from this phase:**
- Sleep mean = 6.81 hrs (below 7-hr clinical recommendation)
- Screen time mean = 7.56 hrs/day (rivals sleep duration)
- Stress mean = 5.43/10 (moderate); anxiety mean = 6.07 (mild threshold)
- GPA slightly left-skewed (mean 3.07, median 3.12)


In [1]:
"""
Phase 1 — Univariate Analysis: Numeric Columns
================================================
Produces histogram + KDE + box plot for each numeric column.
Prints descriptive stats (mean, median, std, skewness, kurtosis).

Outputs: phase1_univariate/figures/[col]_hist.png, [col]_box.png
"""

import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

BASE  = "/Users/joaoquintanilha/Downloads/project-hero/reports/eda_lab"
CLEAN = os.path.join(BASE, "dataset", "student_wellness_clean.csv")
FIGS  = os.path.join(BASE, "phase1_univariate", "figures")
os.makedirs(FIGS, exist_ok=True)

PASTEL = ["#A8C8E8", "#F4A8B0", "#A8E8C8", "#F4D8A8", "#C8A8E8", "#F4F4A8"]
LAYOUT = dict(
    template="plotly_white",
    font=dict(family="Inter, Arial, sans-serif", size=13, color="#4A4A4A"),
    plot_bgcolor="#FAFAFA", paper_bgcolor="#FFFFFF",
    margin=dict(t=80, b=60, l=60, r=40),
)

def save_fig(fig, name):
    fig.write_image(os.path.join(FIGS, name), width=1200, height=600, scale=2)
    print(f"  [saved] {name}")

df = pd.read_csv(CLEAN)

NUMERIC_COLS = [
    ("age",                    "Age",                       "Years",      PASTEL[0]),
    ("gpa",                    "GPA",                       "Grade Points (0–4.0)", PASTEL[1]),
    ("study_hours_per_day",    "Study Hours per Day",       "Hours",      PASTEL[2]),
    ("attendance_rate",        "Class Attendance Rate",     "%",          PASTEL[3]),
    ("sleep_hours_per_night",  "Sleep Hours per Night",     "Hours",      PASTEL[4]),
    ("exercise_days_per_week", "Exercise Days per Week",    "Days",       PASTEL[0]),
    ("screen_time_hours",      "Daily Screen Time",         "Hours",      PASTEL[1]),
    ("social_media_hours",     "Social Media Usage",        "Hours/Day",  PASTEL[2]),
    ("caffeine_mg_per_day",    "Caffeine Intake",           "mg/Day",     PASTEL[3]),
    ("stress_level",           "Stress Level (self-report)","Scale 1–10", PASTEL[4]),
    ("anxiety_score",          "Anxiety Score (GAD-7)",     "0–21",       PASTEL[0]),
    ("depression_score",       "Depression Score (PHQ-9)",  "0–27",       PASTEL[1]),
    ("life_satisfaction",      "Life Satisfaction",         "Scale 1–10", PASTEL[2]),
    ("monthly_spending",       "Monthly Spending",          "USD ($)",    PASTEL[3]),
]

print(f"\n{'='*60}")
print("PHASE 1 — UNIVARIATE ANALYSIS: NUMERIC COLUMNS")
print(f"{'='*60}\n")

summary_rows = []

for col, label, unit, color in NUMERIC_COLS:
    if col not in df.columns:
        continue
    s = df[col].dropna()

    skew = s.skew()
    kurt = s.kurtosis()
    skew_label = ("symmetric" if abs(skew) < 0.5
                  else ("right-skewed" if skew > 0 else "left-skewed"))

    print(f"[{col}]")
    print(f"  mean={s.mean():.2f}  median={s.median():.2f}  std={s.std():.2f}")
    print(f"  min={s.min():.2f}  max={s.max():.2f}  skew={skew:.2f} ({skew_label})  kurtosis={kurt:.2f}")

    summary_rows.append({
        "column": col, "mean": round(s.mean(), 2), "median": round(s.median(), 2),
        "std": round(s.std(), 2), "min": round(s.min(), 2), "max": round(s.max(), 2),
        "skewness": round(skew, 2), "kurtosis": round(kurt, 2),
        "shape": skew_label,
    })

    # ── Histogram with KDE overlay ──
    kde_x = np.linspace(s.min(), s.max(), 300)
    kde_y = stats.gaussian_kde(s)(kde_x)
    kde_y_scaled = kde_y * len(s) * (s.max() - s.min()) / 30  # scale to histogram

    fig_hist = go.Figure()
    fig_hist.add_trace(go.Histogram(
        x=s, nbinsx=30, name="Distribution",
        marker_color=color, opacity=0.75,
        showlegend=True,
    ))
    fig_hist.add_trace(go.Scatter(
        x=kde_x, y=kde_y_scaled, mode="lines",
        line=dict(color="#4A4A4A", width=2, dash="solid"),
        name="KDE", showlegend=True,
    ))
    fig_hist.add_vline(x=s.mean(),   line_dash="dash", line_color="#A8C8E8",
                       annotation_text=f"Mean: {s.mean():.1f}")
    fig_hist.add_vline(x=s.median(), line_dash="dot",  line_color="#F4A8B0",
                       annotation_text=f"Median: {s.median():.1f}")
    fig_hist.update_layout(**LAYOUT,
        title=f"{label} — Distribution",
        xaxis_title=f"{label} ({unit})",
        yaxis_title="Count",
        legend=dict(x=0.78, y=0.92),
    )
    save_fig(fig_hist, f"{col}_hist.png")

    # ── Box plot ──
    fig_box = go.Figure()
    fig_box.add_trace(go.Box(
        y=s, name=label,
        marker_color=color,
        boxpoints="outliers",
        jitter=0.3,
        line=dict(color="#4A4A4A", width=1.5),
    ))
    fig_box.update_layout(**LAYOUT,
        title=f"{label} — Box Plot",
        yaxis_title=f"{label} ({unit})",
        xaxis=dict(showticklabels=False),
    )
    save_fig(fig_box, f"{col}_box.png")

# ── Summary stats CSV ──
summary_df = pd.DataFrame(summary_rows)
summary_path = os.path.join(BASE, "phase1_univariate", "numeric_summary_stats.csv")
summary_df.to_csv(summary_path, index=False)

print(f"\n{'='*60}")
print("NUMERIC UNIVARIATE COMPLETE")
print(f"Figures saved to: {FIGS}")
print(f"Summary stats:    {summary_path}")
print(f"{'='*60}")
print("\nKey highlights:")
print(summary_df[["column", "mean", "median", "skewness", "shape"]].to_string(index=False))



PHASE 1 — UNIVARIATE ANALYSIS: NUMERIC COLUMNS

[age]
  mean=21.49  median=21.00  std=2.27
  min=18.00  max=25.00  skew=0.00 (symmetric)  kurtosis=-1.23


  [saved] age_hist.png


  [saved] age_box.png
[gpa]
  mean=3.07  median=3.12  std=0.45
  min=1.57  max=4.00  skew=-0.38 (symmetric)  kurtosis=0.26


  [saved] gpa_hist.png


  [saved] gpa_box.png
[study_hours_per_day]
  mean=6.88  median=7.00  std=1.79
  min=0.50  max=11.60  skew=-0.23 (symmetric)  kurtosis=0.06


...

  [saved] depression_score_box.png
[life_satisfaction]
  mean=5.43  median=5.40  std=2.36
  min=1.00  max=10.00  skew=0.08 (symmetric)  kurtosis=-0.69


  [saved] life_satisfaction_hist.png


  [saved] life_satisfaction_box.png
[monthly_spending]
  mean=832.56  median=814.60  std=245.34
  min=200.00  max=1709.00  skew=0.33 (symmetric)  kurtosis=0.37


  [saved] monthly_spending_hist.png


  [saved] monthly_spending_box.png

NUMERIC UNIVARIATE COMPLETE
Figures saved to: /Users/joaoquintanilha/Downloads/project-hero/reports/eda_lab/phase1_univariate/figures
Summary stats:    /Users/joaoquintanilha/Downloads/project-hero/reports/eda_lab/phase1_univariate/numeric_summary_stats.csv

Key highlights:
                column   mean  median  skewness     shape
                   age  21.49   21.00      0.00 symmetric
                   gpa   3.07    3.12     -0.38 symmetric
   study_hours_per_day   6.88    7.00     -0.23 symmetric
       attendance_rate  82.54   82.80     -0.25 symmetric
 sleep_hours_per_night   6.81    6.80      0.15 symmetric
exercise_days_per_week   3.41    3.00      0.09 symmetric
     screen_time_hours   7.56    7.60      0.05 symmetric
    social_media_hours   2.55    2.50      0.13 symmetric
   caffeine_mg_per_day 185.25  189.00      0.10 symmetric
          stress_level   5.43    5.40      0.02 symmetric
         anxiety_score   6.07    6.00      0.18 sym

## Part 2 — Categorical Columns

For each categorical column: frequency bar chart sorted descending, showing count and %.
Variables covered: gender, major, year_in_school, has_part_time_job, on_campus.

**Key findings:**
- Gender: balanced 45/45% Male/Female, 10% other identities
- Major: Business (15.2%) and CS (14.1%) largest groups; STEM combined ~41%
- Year: well-balanced across all four years (22–28% each)
- Part-time job: 40.2% of students hold one — large enough for comparison


In [2]:
"""
Phase 1 — Univariate Analysis: Categorical Columns
====================================================
Produces bar charts for each categorical column, sorted by frequency.
Prints value counts and % breakdown.

Outputs: phase1_univariate/figures/[col]_bar.png
"""

import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

BASE  = "/Users/joaoquintanilha/Downloads/project-hero/reports/eda_lab"
CLEAN = os.path.join(BASE, "dataset", "student_wellness_clean.csv")
FIGS  = os.path.join(BASE, "phase1_univariate", "figures")
os.makedirs(FIGS, exist_ok=True)

PASTEL = ["#A8C8E8", "#F4A8B0", "#A8E8C8", "#F4D8A8", "#C8A8E8", "#F4F4A8", "#B8D8F8", "#F8C8D8"]
LAYOUT = dict(
    template="plotly_white",
    font=dict(family="Inter, Arial, sans-serif", size=13, color="#4A4A4A"),
    plot_bgcolor="#FAFAFA", paper_bgcolor="#FFFFFF",
    margin=dict(t=80, b=100, l=60, r=40),
)

def save_fig(fig, name):
    fig.write_image(os.path.join(FIGS, name), width=1200, height=600, scale=2)
    print(f"  [saved] {name}")

df = pd.read_csv(CLEAN)
# on_campus to string for display
df["on_campus_str"] = df["on_campus"].map({True: "On Campus", False: "Off Campus"})

CAT_COLS = [
    ("gender",          "Gender Identity"),
    ("major",           "Academic Major"),
    ("year_in_school",  "Year in School"),
    ("has_part_time_job", "Has Part-Time Job"),
    ("on_campus_str",   "Living Situation (On/Off Campus)"),
]

print(f"\n{'='*60}")
print("PHASE 1 — UNIVARIATE ANALYSIS: CATEGORICAL COLUMNS")
print(f"{'='*60}\n")

for col, label in CAT_COLS:
    counts = df[col].astype(str).value_counts().reset_index()
    counts.columns = ["value", "count"]
    counts["pct"] = (counts["count"] / counts["count"].sum() * 100).round(1)

    print(f"\n[{col}]")
    for _, row in counts.iterrows():
        print(f"  {row['value']:30s}  {row['count']:4d}  ({row['pct']:.1f}%)")

    # sort by count descending
    counts = counts.sort_values("count", ascending=False)

    fig = go.Figure()
    for i, (_, row) in enumerate(counts.iterrows()):
        fig.add_trace(go.Bar(
            x=[row["value"]], y=[row["count"]],
            name=str(row["value"]),
            marker_color=PASTEL[i % len(PASTEL)],
            text=f"{row['count']} ({row['pct']}%)",
            textposition="outside",
            showlegend=False,
        ))

    fig.update_layout(**LAYOUT,
        title=f"{label} — Frequency Distribution",
        xaxis_title=label,
        yaxis_title="Number of Students",
        xaxis=dict(categoryorder="total descending"),
    )
    fname = col.replace("_str", "") + "_bar.png"
    save_fig(fig, fname)

# ── Special: year_in_school with labels ──────────────────────────────────────
year_map = {1: "1st Year (Freshman)", 2: "2nd Year (Sophomore)",
            3: "3rd Year (Junior)", 4: "4th Year (Senior)"}
df["year_label"] = df["year_in_school"].map(year_map)
year_counts = df["year_label"].value_counts().reset_index()
year_counts.columns = ["year", "count"]
year_counts["pct"] = (year_counts["count"] / year_counts["count"].sum() * 100).round(1)
year_order = ["1st Year (Freshman)", "2nd Year (Sophomore)", "3rd Year (Junior)", "4th Year (Senior)"]

fig = px.bar(year_counts, x="year", y="count",
             color="year", color_discrete_sequence=PASTEL,
             text=year_counts["count"].astype(str) + "<br>(" + year_counts["pct"].astype(str) + "%)",
             category_orders={"year": year_order},
             labels={"year": "Academic Year", "count": "Number of Students"})
fig.update_traces(textposition="outside", showlegend=False)
fig.update_layout(**LAYOUT, title="Distribution by Academic Year")
fig.write_image(os.path.join(FIGS, "year_in_school_labeled_bar.png"), width=1200, height=600, scale=2)
print(f"  [saved] year_in_school_labeled_bar.png")

print(f"\n{'='*60}")
print("CATEGORICAL UNIVARIATE COMPLETE")
print(f"Figures saved to: {FIGS}")
print(f"{'='*60}")



PHASE 1 — UNIVARIATE ANALYSIS: CATEGORICAL COLUMNS


[gender]
  Male                             240  (45.1%)
  Female                           240  (45.1%)
  Non-binary                        33  (6.2%)
  Prefer not to say                 19  (3.6%)


  [saved] gender_bar.png

[major]
  Business                          81  (15.2%)
  Computer Science                  75  (14.1%)
  Psychology                        71  (13.3%)
  Biology                           54  (10.2%)
  Political Science                 48  (9.0%)
  Nursing                           47  (8.8%)
  Mechanical Engineering            43  (8.1%)
  Art & Design                      39  (7.3%)
  Communications                    39  (7.3%)
  Economics                         35  (6.6%)


  [saved] major_bar.png

[year_in_school]
  1                                148  (27.8%)
  2                                140  (26.3%)
  3                                125  (23.5%)
  4                                119  (22.4%)


  [saved] year_in_school_bar.png

[has_part_time_job]
  No                               318  (59.8%)
  Yes                              214  (40.2%)


  [saved] has_part_time_job_bar.png

[on_campus_str]
  On Campus                        291  (54.7%)
  Off Campus                       241  (45.3%)


  [saved] on_campus_bar.png


  [saved] year_in_school_labeled_bar.png

CATEGORICAL UNIVARIATE COMPLETE
Figures saved to: /Users/joaoquintanilha/Downloads/project-hero/reports/eda_lab/phase1_univariate/figures


## Summary

All 28+ figures saved to `phase1_univariate/figures/`.

**Top 3 variables flagged for Phase 2 hypothesis generation:**
1. `sleep_hours_per_night` — below clinical recommendation, high variance
2. `screen_time_hours` — enormous daily commitment (7.56 hrs avg), wide spread
3. `stress_level` — central wellness axis; STEM vs. non-STEM contrast expected

See `phase1_univariate/report.md` for full narrative interpretation of every variable.
